# Adversarial validation: can a recurrent model tell training posts from test posts?

A classifier is trained to separate **test-like** texts (the 477 test posts + the 26 labeled rows) from **training** posts. Its out-of-fold AUC measures the shift:

- AUC ≈ 0.5: the two sets look alike, so validation on the training data should predict the leaderboard.
- AUC → 1.0: the sets are easy to tell apart; validation is optimistic and the model needs test-like training data.

As the assignment requires, the classifier uses **the same recurrent encoder as the main task** (the same BPE tokenizer, embedding and 2-layer bidirectional GRU, 256/256); only the output head differs (mean-pool + one logit).
It is run for several training-data variants, and for a **control** (two random halves of the same training data, which must give AUC ≈ 0.5), so the effect of the test-like augmentation can be measured.

In [1]:
# ============================ 1. CONFIG ============================
VARIANTS = {"long_all":      "data/splits/long_all.parquet",
            "long_aug_all":  "data/splits/long_aug_all.parquet",
            "long_aug2_all": "data/splits/long_aug2_all.parquet"}      # from make_aug2.ipynb; drop it if you have not made it
TEST_CSV      = "Data/test.csv"
LABELING_FILE = "data/splits/val_labeling_sample.parquet"
BPE_JSON      = "sensitive_rua_bpe_tokenizer.json"     # case-sensitive; the first run used the lowercasing rua_bpe_tokenizer.json

N_NEG     = 6000          # training posts per variant (negatives)
N_FOLDS   = 5
SEED      = 42
MAX_TOK   = 512           # tokens kept per post
# same encoder as the main task:
EMB, HID, LAYERS, BIDIR = 256, 256, 2, True
# optimisation (same family of settings as the main task)
EPOCHS, BATCH, LR, WEIGHT_DECAY, DROPOUT, CLIP_NORM = 6, 32, 2e-3, 1e-2, 0.2, 1.0

USE_WANDB, WANDB_ENTITY, WANDB_PROJECT = True, "aboichuk06-kyiv-school-of-economics", "rnn"

In [2]:
# ============================ 2. SETUP ============================
import math, random, re, time
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from tokenizers import Tokenizer

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("device:", DEVICE)

tk = Tokenizer.from_file(BPE_JSON); tk.no_truncation()
PAD, VOCAB = tk.token_to_id("<pad>"), tk.get_vocab_size()

def encode(texts):
    return [np.asarray(e.ids[:MAX_TOK], np.int64) for e in tk.encode_batch(list(texts))]

def auc(y, s):
    """Rank-based ROC AUC (ties get average rank)."""
    y = np.asarray(y); r = pd.Series(s).rank().to_numpy()
    n1 = int(y.sum()); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def folds_stratified(y, k, seed):
    """fold id per sample, stratified by label."""
    rng = np.random.default_rng(seed); f = np.zeros(len(y), int)
    for c in (0, 1):
        idx = np.flatnonzero(y == c); rng.shuffle(idx); f[idx] = np.arange(len(idx)) % k
    return f

device: cuda


In [3]:
# ============================ 3. DATA ============================
test = pd.read_csv(TEST_CSV); test["text"] = test["text"].astype(str)
lab = pd.read_parquet(LABELING_FILE)
pos_texts = test["text"].tolist() + lab["text"].astype(str).tolist()      # "test-like" = label 1
print(len(pos_texts), "test-like texts (test + labeled sample)")

def sample_train(path, n, seed):
    d = pd.read_parquet(path, columns=["text", "split"])
    return d[d["split"] == "dev"]["text"].sample(n=n, random_state=seed).tolist()

503 test-like texts (test + labeled sample)


In [4]:
# ============================ 4. MODEL: the main task's encoder + a pooled head ============================
class SeqClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, EMB, padding_idx=PAD)
        self.drop = nn.Dropout(DROPOUT)
        self.rnn = nn.GRU(EMB, HID, num_layers=LAYERS, batch_first=True, bidirectional=BIDIR,
                          dropout=DROPOUT if LAYERS > 1 else 0.0)
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(HID * (2 if BIDIR else 1), 1))
    def forward(self, ids, lens):
        x = self.drop(self.emb(ids))
        y, _ = self.rnn(pack_padded_sequence(x, lens, batch_first=True, enforce_sorted=False))
        y, _ = pad_packed_sequence(y, batch_first=True, total_length=ids.size(1))
        mask = (torch.arange(ids.size(1), device=ids.device)[None] < lens.to(ids.device)[:, None]).unsqueeze(-1)
        pooled = (y * mask).sum(1) / mask.sum(1).clamp(min=1)               # masked mean over time
        return self.head(pooled).squeeze(-1)

def batches(ids, idx, bs, shuffle, rng):
    idx = np.array(idx)
    if shuffle: rng.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]; L = [len(ids[j]) for j in b]; m = max(L)
        x = np.full((len(b), m), PAD, np.int64)
        for r, j in enumerate(b): x[r, :L[r]] = ids[j]
        yield b, torch.from_numpy(x).to(DEVICE), torch.tensor(L)

@torch.no_grad()
def predict(model, ids, idx):
    model.eval(); out = np.zeros(len(ids))
    order = sorted(idx, key=lambda j: len(ids[j]))
    for b, x, L in batches(ids, order, 128, False, None):
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=AMP):
            out[b] = torch.sigmoid(model(x, L).float()).cpu().numpy()
    return out

def run_adversarial(neg_texts, pos_texts, seed, tag):
    """5-fold out-of-fold AUC of 'is this a test-like post?'"""
    texts = list(neg_texts) + list(pos_texts)
    y = np.array([0] * len(neg_texts) + [1] * len(pos_texts))
    ids = encode(texts); fold = folds_stratified(y, N_FOLDS, seed)
    oof = np.zeros(len(y)); t0 = time.time()
    for k in range(N_FOLDS):
        set_seed(seed + k)
        tr, va = np.flatnonzero(fold != k), np.flatnonzero(fold == k)
        model = SeqClassifier().to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        steps = EPOCHS * math.ceil(len(tr) / BATCH); step = 0
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))
        pw = torch.tensor((y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1), device=DEVICE)
        rng = np.random.default_rng(seed + k)
        for ep in range(EPOCHS):
            model.train()
            for b, x, L in batches(ids, tr, BATCH, True, rng):
                with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=AMP):
                    logit = model(x, L)
                loss = nn.functional.binary_cross_entropy_with_logits(logit.float(), torch.from_numpy(y[b]).float().to(DEVICE), pos_weight=pw)
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM); opt.step(); sched.step()
        oof[va] = predict(model, ids, va)[va]
        print(f"  [{tag}] fold {k}: AUC {auc(y[va], oof[va]):.4f} ({(time.time()-t0)/60:.1f} min)")
    return auc(y, oof), texts, y, oof

In [5]:
# ============================ 5. RUN ============================
import wandb
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="adversarial_validation",
                 mode="online" if USE_WANDB else "disabled",
                 config=dict(n_neg=N_NEG, emb=EMB, hid=HID, layers=LAYERS, epochs=EPOCHS, lr=LR, folds=N_FOLDS))

results, keep = {}, {}
for name, path in VARIANTS.items():
    neg = sample_train(path, N_NEG, SEED)
    a, texts, y, oof = run_adversarial(neg, pos_texts, SEED, name)
    results[name] = a; keep[name] = (texts, y, oof)
    print(f"{name}: AUC {a:.4f}\n")

# control: two random halves of the same training data must NOT be separable
first = list(VARIANTS.values())[0]
ctrl = sample_train(first, 2 * len(pos_texts), SEED + 1)
a_ctrl, *_ = run_adversarial(ctrl[:len(pos_texts)], ctrl[len(pos_texts):], SEED, "control")
results["control (train vs train)"] = a_ctrl
print(pd.Series(results).round(4).rename("AUC").to_frame())
run.summary.update({f"auc/{k}": v for k, v in results.items()}); run.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\aboic\_netrc.
wandb: Currently logged in as: aboichuk06 (aboichuk06-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 35y8v10j
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in C:\Users\aboic\IdeaProjects\language_model\DL_ass3\wandb\run-20261109_164528-35y8v10j
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run adversarial_validation
wandb:  View project at https://wandb.ai/aboichuk06-kyiv-school-of-economics/rnn
wandb:  View run at https://wandb.ai/aboichuk06-kyiv-school-of-economics/rnn/runs/35y8v10j


  [long_all] fold 0: AUC 0.9708 (0.9 min)
  [long_all] fold 1: AUC 0.9824 (1.7 min)
  [long_all] fold 2: AUC 0.9868 (2.5 min)
  [long_all] fold 3: AUC 0.9859 (3.4 min)
  [long_all] fold 4: AUC 0.9923 (4.2 min)
long_all: AUC 0.9811

  [long_aug_all] fold 0: AUC 0.9753 (0.9 min)
  [long_aug_all] fold 1: AUC 0.9866 (1.7 min)
  [long_aug_all] fold 2: AUC 0.9822 (2.5 min)
  [long_aug_all] fold 3: AUC 0.9826 (3.4 min)
  [long_aug_all] fold 4: AUC 0.9901 (4.2 min)
long_aug_all: AUC 0.9831

  [long_aug2_all] fold 0: AUC 0.9738 (0.9 min)
  [long_aug2_all] fold 1: AUC 0.9901 (1.7 min)
  [long_aug2_all] fold 2: AUC 0.9848 (2.5 min)
  [long_aug2_all] fold 3: AUC 0.9803 (3.4 min)
  [long_aug2_all] fold 4: AUC 0.9894 (4.3 min)
long_aug2_all: AUC 0.9826

  [control] fold 0: AUC 0.4473 (0.1 min)
  [control] fold 1: AUC 0.5551 (0.3 min)
  [control] fold 2: AUC 0.4895 (0.4 min)
  [control] fold 3: AUC 0.5361 (0.5 min)


wandb: updating run metadata


  [control] fold 4: AUC 0.5151 (0.7 min)
                             AUC
long_all                  0.9811
long_aug_all              0.9831
long_aug2_all             0.9826
control (train vs train)  0.5076


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log
wandb: 
wandb: Run summary:
wandb: auc/control (train vs train) 0.50762
wandb:                 auc/long_all 0.9811
wandb:            auc/long_aug2_all 0.98265
wandb:             auc/long_aug_all 0.98308
wandb: 
wandb:  View run adversarial_validation at: https://wandb.ai/aboichuk06-kyiv-school-of-economics/rnn/runs/35y8v10j
wandb:  View project at: https://wandb.ai/aboichuk06-kyiv-school-of-economics/rnn
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: .\wandb\run-20261109_164528-35y8v10j\logs


In [6]:
# ============================ 6. WHAT GIVES THE TEST AWAY? ============================
# (a) tokens that are much more frequent in test posts than in training posts
PAT = re.compile(r"\w+|[^\w\s]")
def tok_doc_freq(texts):
    c = Counter()
    for t in texts: c.update(set(m.group().lower() for m in PAT.finditer(t)))
    return c
ct = tok_doc_freq(pos_texts)
for vname in VARIANTS:
    tr_texts = keep[vname][0][:N_NEG]
    cn = tok_doc_freq(tr_texts)
    rows = [(w, c / len(pos_texts), cn.get(w, 0) / len(tr_texts)) for w, c in ct.items() if c >= 15]
    tab = pd.DataFrame(rows, columns=["token", "share_of_test_posts", "share_of_train_posts"])
    tab["ratio"] = (tab["share_of_test_posts"] + 1e-3) / (tab["share_of_train_posts"] + 1e-3)
    print(f"\nmost test-specific tokens vs '{vname}' (document frequency):")
    print(tab.sort_values("ratio", ascending=False).head(20).round(3).to_string(index=False))
first_name = list(VARIANTS)[-1]            # the last (richest) variant is used for the example posts below

# (b) the training posts the classifier finds most test-like, and the test posts it finds least test-like
texts, y, oof = keep[first_name]
tr_idx = np.flatnonzero(y == 0); te_idx = np.flatnonzero(y == 1)
print("\ntraining posts that look most like the test:")
for j in tr_idx[np.argsort(-oof[tr_idx])[:4]]: print(f"  {oof[j]:.2f}  {texts[j][:110]!r}")
print("\ntest posts that look most like the training data:")
for j in te_idx[np.argsort(oof[te_idx])[:4]]: print(f"  {oof[j]:.2f}  {texts[j][:110]!r}")


most test-specific tokens vs 'long_all' (document frequency):
        token  share_of_test_posts  share_of_train_posts   ratio
            |                0.129                 0.000 130.225
            👈                0.095                 0.000  96.427
  підписатися                0.083                 0.000  84.499
    надіслати                0.083                 0.000  72.428
            ㅤ                0.066                 0.000  66.606
       tiktok                0.054                 0.000  54.678
  підписатись                0.054                 0.000  54.678
            👉                0.052                 0.000  52.690
      дарниця                0.048                 0.000  48.714
         livе                0.048                 0.000  48.714
            🔷                0.048                 0.000  48.714
запропонувати                0.078                 0.001  47.121
            🫶                0.046                 0.000  46.726
      хуйовий              

In [7]:
# ============================ 7. SURFACE OR CONTENT?  (both sides cleaned of boilerplate) ============================
# Strip the cheap giveaways from BOTH sides with the same function: lines repeated >= 3 times in the test posts
# (channel footers), URLs, #hashtags, @handles, emoji and invisible characters. If the AUC is still high, the
# remaining shift is vocabulary, topic and style (e.g. Kyiv streets and districts), which augmentation cannot fake.
URL = re.compile(r"https?://\S+|t\.me/\S+")
TAG = re.compile(r"(?<!\w)[#@]\w+")
SYM = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\u2190-\u21FF\uFE0F\u200d\u3164\u00a0]")
line_cnt = Counter(l.strip() for t in pos_texts for l in t.split("\n") if 0 < len(l.strip()) <= 70)
REPEATED = {l for l, c in line_cnt.items() if c >= 3}
print(len(REPEATED), "repeated test lines treated as boilerplate, e.g.", sorted(REPEATED, key=lambda l: -line_cnt[l])[:4])

def clean(t):
    t = URL.sub(" ", t)
    t = "\n".join(l for l in t.split("\n") if l.strip() not in REPEATED)
    t = TAG.sub(" ", SYM.sub(" ", t))
    t = re.sub(r"[ \t]+", " ", t); t = re.sub(r"\n\s*\n+", "\n\n", t)
    return t.strip()

pos_clean = [c for c in map(clean, pos_texts) if len(c) >= 20]
neg_clean = [c for c in map(clean, sample_train(list(VARIANTS.values())[0], 3 * N_NEG, SEED + 7)) if len(c) >= 20][:N_NEG]
print(f"{len(pos_clean)} test-like posts and {len(neg_clean)} training posts left after cleaning")
a_clean, *_ = run_adversarial(neg_clean, pos_clean, SEED, "cleaned")
print(f"\nAUC with boilerplate removed on both sides: {a_clean:.4f}   (raw long_all: {results['long_all']:.4f})")
print("  < 0.65: the shift is almost entirely surface boilerplate -> augmentation can close it\n"
      "  0.65-0.85: part content/style\n"
      "  > 0.85: mostly content and style (topics, Kyiv toponyms, how people write): augmentation of boilerplate will not be enough")

26 repeated test lines treated as boilerplate, e.g. ['ㅤ', 'Підписатись на Дарниця Livе', '👉👉Запропонувати новину👈👈', 'Хуйовий Київ | Надіслати новину']
484 test-like posts and 6000 training posts left after cleaning
  [cleaned] fold 0: AUC 0.9759 (0.8 min)
  [cleaned] fold 1: AUC 0.9767 (1.7 min)
  [cleaned] fold 2: AUC 0.9764 (2.5 min)
  [cleaned] fold 3: AUC 0.9627 (3.3 min)
  [cleaned] fold 4: AUC 0.9876 (4.2 min)

AUC with boilerplate removed on both sides: 0.9727   (raw long_all: 0.9811)
  < 0.65: the shift is almost entirely surface boilerplate -> augmentation can close it
  0.65-0.85: part content/style
  > 0.85: mostly content and style (topics, Kyiv toponyms, how people write): augmentation of boilerplate will not be enough


## Reading the result

- **Control ≈ 0.5** confirms the pipeline is not leaking (same sanity check as the previous assignment).
- **AUC of `long_all`** is the size of the covariate shift. A high value explains why CV ≈ 0.91 but the leaderboard ≈ 0.55.
- **AUC of `long_aug_all` should be lower**: the augmentation removes the cheapest giveaways (footers, emoji, hashtags). What remains is content/style that augmentation does not cover; the token table in cell 6 shows what it is.
- If AUC stays high, it is not a bug: it says the training news text and Telegram posts differ in vocabulary and style, and no validation on news will track the leaderboard.